In [5]:
!pip install -q sentence-transformers faiss-cpu transformers torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 18.0 MB/s eta 0:00:00


In [7]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Knowledge base
documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]

# 2. Embed documents
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embed_model.encode(documents)

# 3. Build FAISS index
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(doc_embeddings).astype("float32")
)

# 4. Query
query = "What is RAG in AI?"

query_embedding = embed_model.encode([query])

# 5. Retrieve top-2 relevant chunks
D, I = index.search(
    np.array(query_embedding).astype("float32"),
    k=2
)

retrieved_chunks = [documents[i] for i in I[0]]

# 6. Build augmented prompt
context = " ".join(retrieved_chunks)

prompt = f"""Context: {context}

Question: {query}

Answer:"""

# 7. Load FLAN-T5
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 8. Tokenize prompt
inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

# 9. Generate answer
outputs = model.generate(
    **inputs,
    max_new_tokens=60
)

# 10. Decode answer
answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

# 11. Display results
print("Retrieved Context:")

for chunk in retrieved_chunks:
    print("-", chunk)

print("\nAnswer:")
print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Retrieved Context:
- Python is a popular high-level programming language used in AI development.
- Retrieval-Augmented Generation combines document retrieval with text generation.

Answer:
combines document retrieval with text generation
